## Summarizing a text using an LLM

As an LLM "understands" a language, it can be suited for tasks like translation or summarization.

In this Notebook, we are going to use our LLM to summarize some texts, especially claims examples.

### Requirements and Imports

If you have selected the right workbench image to launch as per the Lab's instructions, you should already have all the needed libraries. If not uncomment the first line in the next cell to install all the right packages.

In [1]:
# Uncomment the following line only if you have not selected the right workbench image, or are using this notebook outside of the workshop environment.
# !pip install langchain==0.3.25 langchain-openai==0.3.22 langchain-core==0.3.65

import json
import os
from os import listdir
from os.path import isfile, join

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

### Langchain pipeline

Again, we are going to use Langchain to define our summarization pipeline.

In [2]:
# LLM Inference Server URL
inference_server_url = "https://redhataimistral-small-quantizedw4a16-mistral.apps.cluster-d44sv.d44sv.sandbox2961.opentlc.com"

# LLM definition
llm = ChatOpenAI(
    openai_api_key="xxx", # replace xxx with your own token!!!
    openai_api_base=f"{inference_server_url}/v1",
    model_name="redhataimistral-small-quantizedw4a16",
    temperature=0.01,
    max_tokens=512,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
    top_p=0.9,
    presence_penalty=0.5,
    model_kwargs={
        "stream_options": {"include_usage": True}
    }
)

The **template** we will use is now formatted for this specific summarization task.

In [3]:
template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        """Opsummer følgende tekst på dansk, som fortæller om, hvad agenten har snakket med kunden om. 
        Det skal være venligt og professionelt, så den kan videresendes til kunden.
        Den skal altid slutte af med: 'Med venlig hilsen' og så navnet på agenten.
        """),
    HumanMessagePromptTemplate.from_template(
        """### TEXT:
        {input}
        ### SUMMARY:"""
    )
])

In [4]:

summary_input = f"""
    Kunde: Hej, jeg vil gerne anmelde et biluheld og starte en skadeanmeldelse.
    
    Agent: Selvfølgelig — det er jeg ked af at høre. Jeg skal nok hjælpe dig. Må jeg få dit policenummer?
    
    Kunde: Ja, det er 45678231.
    
    Agent: Tak. Kan du kort forklare, hvad der skete?
    
    Kunde: Jeg kørte i går, og en anden bil kørte ind i mig bagfra ved et lyskryds. Ingen kom til skade, men bagkofangeren er ødelagt.
    
    Agent: Det er godt at høre, at ingen kom til skade. Kom politiet til stedet?
    
    Kunde: Ja, og jeg har en politirapport.
    
    Agent: Perfekt. Vi får brug for rapporten og billeder af skaden. Har du også informationer om den anden fører?
    
    Kunde: Ja, jeg udvekslede oplysninger. Jeg har navn, telefonnummer, nummerplade og forsikringsselskab.
    
    Agent: Super. Jeg åbner din sag nu. Du får snart en e-mail med instruktioner til at uploade dokumenter og billeder. Har du brug for en lejebil, mens din bil bliver repareret?
    
    Kunde: Ja, det har jeg.
    
    Agent: Det klarer vi. Er der ellers noget, jeg kan hjælpe dig med?
    
    Kunde: Nej, det var alt. Tak.
    
    Agent: Selv tak. Vi kontakter dig snart. Hav en god dag!
"""
prompt = template.invoke({"input": summary_input})
resp = llm.invoke(input=prompt)


### Opsummering af samtale

Hej [Kundens navn],

Jeg har i dag snakket med dig om anmeldelsen af et biluheld, og jeg vil gerne opsummere, hvad vi har diskuteret.

Du har anmeldt et uheld, hvor en anden bil kørte ind i dig bagfra ved et lyskryds. Heldigvis kom ingen til skade, men bagkofangeren på din bil er ødelagt. Du har en politirapport og har udvekslet oplysninger med den anden fører, herunder navn, telefonnummer, nummerplade og forsikringsselskab.

Vi har åbnet din sag, og du vil snart modtage en e-mail med instruktioner til at uploade dokumenter og billeder af skaden. Vi vil også sørge for, at du får en lejebil, mens din bil bliver repareret.

Hvis der er noget andet, du har brug for hjælp til, er du velkommen til at kontakte os igen.

Med venlig hilsen,

[Agentens navn]

You can come back to this notebook at section 3.7 for some optional exercises if you want.